### DATA INGESTION

In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content= "This is a Document",
    metadata = {
        "source" : "example.txt",
        "pages" : 1,
        "author" : "Abrar Bari",
        "date_created" : "2026-08-01"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Abrar Bari', 'date_created': '2026-08-01'}, page_content='This is a Document')

In [3]:
## creating a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
sample_texts = {
    "../data/text_files/python_intro.txt" : """
    Python is a programming language that lets you work quickly and integrate systems more effectively.
    Python is a high-level, interpreted programming language known for its simplicity and readability.
    Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
    programming languages in the world.

    Key Features :
    Easy to learn and use
    Extensive standard library
    Cross-platform compatibility
    Strong community support

    Python is widely used in web development, data science, artificial intelligence, and automation.
    
    """,
    "../data/text_files/machine_learning.txt" :
    """
    Machine learning (ML) is the study of computer algorithms that improve automatically through experience.
    It is seen as a subset of artificial intelligence (AI). Machine learning focuses on the development of algorithms and mathematical models to recognize patterns in data, learn from it, 
    and make predictions or decisions without being explicitly programmed to do so.

    Key Features : 
    Supervised learning
    Unsupervised learning
    Reinforcement learning
    Deep learning    

    Machine learning has applications in various fields, including computer vision, natural language processing, recommendation systems, and more.
    """
}

In [5]:
for filepath, content in sample_texts.items():
    with open(filepath, "w") as f:
        f.write(content)    
print(f"Created {filepath}")    

Created ../data/text_files/machine_learning.txt


In [6]:
### Text Loader
from langchain_community.document_loaders import TextLoader

c:\Code\RAG_Traditional\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
loader = TextLoader(
    "../data/text_files/python_intro.txt",
    encoding="utf8"
)
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='\n    Python is a programming language that lets you work quickly and integrate systems more effectively.\n    Python is a high-level, interpreted programming language known for its simplicity and readability.\n    Created by Guido van Rossum and first released in 1991, Python has become one of the most popular\n    programming languages in the world.\n\n    Key Features :\n    Easy to learn and use\n    Extensive standard library\n    Cross-platform compatibility\n    Strong community support\n\n    Python is widely used in web development, data science, artificial intelligence, and automation.\n\n    ')]


In [8]:
### Directory loader
from langchain_community.document_loaders import DirectoryLoader
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls = TextLoader,
    loader_kwargs={"encoding":"utf8"},
    show_progress=False
)
documents = dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='\n    Machine learning (ML) is the study of computer algorithms that improve automatically through experience.\n    It is seen as a subset of artificial intelligence (AI). Machine learning focuses on the development of algorithms and mathematical models to recognize patterns in data, learn from it, \n    and make predictions or decisions without being explicitly programmed to do so.\n\n    Key Features : \n    Supervised learning\n    Unsupervised learning\n    Reinforcement learning\n    Deep learning    \n\n    Machine learning has applications in various fields, including computer vision, natural language processing, recommendation systems, and more.\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='\n    Python is a programming language that lets you work quickly and integrate systems more effectively.\n    Python is a high-level, interpreted prog

In [9]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls = PyMuPDFLoader,
    show_progress=False
)
pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'dvips + GPL Ghostscript GIT PRERELEASE 9.08', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-04-12T20:43:27-04:00', 'source': '..\\data\\pdf\\1409.1556v6.pdf', 'file_path': '..\\data\\pdf\\1409.1556v6.pdf', 'total_pages': 14, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2015-04-12T20:43:27-04:00', 'trapped': '', 'modDate': "D:20150412204327-04'00'", 'creationDate': "D:20150412204327-04'00'", 'page': 0}, page_content='arXiv:1409.1556v6  [cs.CV]  10 Apr 2015\nPublished as a conference paper at ICLR 2015\nVERY DEEP CONVOLUTIONAL NETWORKS\nFOR LARGE-SCALE IMAGE RECOGNITION\nKaren Simonyan∗& Andrew Zisserman+\nVisual Geometry Group, Department of Engineering Science, University of Oxford\n{karen,az}@robots.ox.ac.uk\nABSTRACT\nIn this work we investigate the effect of the convolutional network depth on its\naccuracy in the large-scale image recognition setting. Our main contribution is\na thorough

In [10]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

### Embeddings and VectorStoreDB

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer  
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
class EmbeddingManager:
    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}") 
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded. Please call _load_model() first.")
        print(f"Generating embeddings for {len(texts)}...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings
    def get_embedding_dimension(self) -> int:
        if not self.model:
            raise ValueError("Model not loaded.")
        return self.model.get_sentence_embedding_dimension()

embedding_manager = EmbeddingManager()
embedding_manager

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 962.92it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Embedding dimension: 384


In [23]:
class VectorStore:
    def __init__(self, collection_name: str="pdf_documents", persist_directory: str = "../data/vector_store" ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata = {
                    "description": "PDF Documents"
                }
            )
            print(f"Store initialized: {self.collection_name}")
            print(f"Existing documents:, {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must be the same.")
        print(f"Adding {len(documents)} documents to the store...")
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
 
            metadata = dict(doc.metadata)
            metadata["doc_id"] = doc_id
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                documents = documents_text,
                metadatas = metadatas
            )
            print(f"Added {len(documents)} documents to the store.")
            print(f"Existing documents:, {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the store: {e}")
            raise   

vectorstore = VectorStore()
vectorstore 

Store initialized: pdf_documents
Existing documents:, 212


In [24]:
chunks = split_documents(pdf_documents)
chunks

Split 21 documents into 106 chunks

Example chunk:
Content: arXiv:1409.1556v6  [cs.CV]  10 Apr 2015
Published as a conference paper at ICLR 2015
VERY DEEP CONVOLUTIONAL NETWORKS
FOR LARGE-SCALE IMAGE RECOGNITION
Karen Simonyan∗& Andrew Zisserman+
Visual Geomet...
Metadata: {'producer': 'dvips + GPL Ghostscript GIT PRERELEASE 9.08', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-04-12T20:43:27-04:00', 'source': '..\\data\\pdf\\1409.1556v6.pdf', 'file_path': '..\\data\\pdf\\1409.1556v6.pdf', 'total_pages': 14, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2015-04-12T20:43:27-04:00', 'trapped': '', 'modDate': "D:20150412204327-04'00'", 'creationDate': "D:20150412204327-04'00'", 'page': 0}


[Document(metadata={'producer': 'dvips + GPL Ghostscript GIT PRERELEASE 9.08', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-04-12T20:43:27-04:00', 'source': '..\\data\\pdf\\1409.1556v6.pdf', 'file_path': '..\\data\\pdf\\1409.1556v6.pdf', 'total_pages': 14, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2015-04-12T20:43:27-04:00', 'trapped': '', 'modDate': "D:20150412204327-04'00'", 'creationDate': "D:20150412204327-04'00'", 'page': 0}, page_content='arXiv:1409.1556v6  [cs.CV]  10 Apr 2015\nPublished as a conference paper at ICLR 2015\nVERY DEEP CONVOLUTIONAL NETWORKS\nFOR LARGE-SCALE IMAGE RECOGNITION\nKaren Simonyan∗& Andrew Zisserman+\nVisual Geometry Group, Department of Engineering Science, University of Oxford\n{karen,az}@robots.ox.ac.uk\nABSTRACT\nIn this work we investigate the effect of the convolutional network depth on its\naccuracy in the large-scale image recognition setting. Our main contribution is\na thorough

In [25]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks,embeddings)  

Generating embeddings for 106...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.76it/s]


Adding 106 documents to the store...
Added 106 documents to the store.
Existing documents:, 318


### Retrieval

In [26]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """          
        Retrieve relevant documents for a query 
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [27]:
rag_retriever

In [28]:
rag_retriever.retrieve("What is Yolov5")

Retrieving documents for query: 'What is Yolov5'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1...


Batches: 100%|██████████| 1/1 [00:00<00:00, 99.99it/s]

Retrieved 5 documents (after filtering)


[{'id': 'doc_7f4ab1b4_91',
  'content': 'Fig. 5. Real time detection output of our YOLOv8 model.\nFig. 6. Frames of real time moving sign language classes from v8 model\nFig. 7. F1 confidence curve and precision graph of YOLOv8 model.\nFig. 8. Validation graphs of YOLOv8 model.\nTest results of YOLOv5 Model:\nHere, we ran 100 epochs for our YOLOv5 model with the total\nof 22 classes.In fig. 9, real time detection of some classes is\nshown.\nFig. 9. Real time detection output of our YOLOv5 model.\nFig. 10. F1 confidence curve and precision graph of YOLOv5 model\nFrom fig. 10, it is seen that the confidence curve for the v5\nmodel is 69.4 percent and the precision level is 92.8 percent\nfor all classes.\nFig. 11. Validation graphs of YOLOv5 model.\nFrom fig. 11, the training and validation box loss, object loss,\nclass loss, precision metrics and recall metrics consecutively\nfor the v5 model is shown. As we ran the model for 100\nepochs. The curve is much smoother than the v8 model.\n2)

In [29]:
rag_retriever.retrieve("The ConvNet configurations")

Retrieving documents for query: 'The ConvNet configurations'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1...


Batches: 100%|██████████| 1/1 [00:00<00:00, 107.93it/s]

Retrieved 5 documents (after filtering)


[{'id': 'doc_1b433612_5',
  'content': 'Published as a conference paper at ICLR 2015\nconﬁgurations are compared on the ILSVRC classiﬁcation task in Sect. 4. Sect. 5 concludes the\npaper. For completeness, we also describe and assess our ILSVRC-2014 object localisation system\nin Appendix A, and discuss the generalisation of very deep features to other datasets in Appendix B.\nFinally, Appendix C contains the list of major paper revisions.\n2\nCONVNET CONFIGURATIONS\nTo measure the improvement brought by the increased ConvNet depth in a fair setting, all our\nConvNet layer conﬁgurations are designed using the same principles, inspired by Ciresan et al.\n(2011); Krizhevsky et al. (2012). In this section, we ﬁrst describe a generic layout of our ConvNet\nconﬁgurations (Sect. 2.1) and then detail the speciﬁc conﬁgurations used in the evaluation (Sect. 2.2).\nOur design choices are then discussed and compared to the prior art in Sect. 2.3.\n2.1\nARCHITECTURE',
  'metadata': {'title': '',
 

In [51]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",   
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.2,
    max_output_tokens=1024
)

In [53]:
def rag_simple(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        return "No relevant documents found."
    prompt = f"""Use the following context to answer the question concisely.
    Context: {context}
    Question: {query}
    Answer:"""
    response = llm.invoke(prompt)    
    return response.content

In [54]:
answer = rag_simple("What is Yolov5?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is Yolov5?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1...


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.15it/s]

Retrieved 3 documents (after filtering)


[{'type': 'text', 'text': 'Based on the context, YOLOv5 is a **real-time detection model** that was trained for 100 epochs on 22 classes. It achieved a confidence curve of 69.4%, a precision level of 92.8%, and produced smoother validation curves than the YOLOv8 model.', 'extras': {'signature': 'EsANCr0NAb4+9vuGOhaMqTEASLegTUFSOWsJK8TFtE72WyLBuSe7BkVemem21MOsaMb9d2tjPXe6PfvHZuxk4NKUQ1YRR6Pz8l80qcZekyvMDWOLENVjB28GYAVyYZAzN95hmVD+f50yJsVfG7QajjuGzNJrkvDCHSLnXbOmwdkssAEc1PHb2dpzlYWAlZFpWVJHL9oF7BFIwpRppOndlihOYvlVW7msV16LJreuriCFK/+Nm1Oi5Tp+nNfopOUzvhgjd2NCuKi5RheJ3IB97scZjCiR8w7zDmQxgbFthqlLCCKnPv1Y2rZPlwR1CprCFwuQrkCJer3a9m/m5PmITpPLiFOBe7D/myvPku4G07eUQtpqmhJVJG6aCafkGU5pWwz78URN2X+sGQVmzxKSZXu6yxGB7ERLdME6fy5onpGM3XkZHosxahUhlQCycEQvrQsRPwQoa/RbzBzUd/eT5VXOgx7Oqq81Xxse4rGrowg1PAFCr8GKj3qcOb4tT1L4HwAQswkgXWdp6d9d8P+qnPAFKD/h19wCo3QzuUS8MXIqHzPr9V9Fsbqb2nE4aVbwL3zk1sIXU+PiK0S5o5R+nFbpil0Bfsb8IpUBGNLgLjTaZdmM8UYNe3vTZyDmrXIihP+MHcSvVkLt6oadVJy72qEto/Qe+7GOpwp/Lh/D2CVpaNCnJtpZvG74ypUuMZt

In [58]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Yolov5", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Yolov5'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1...


Batches: 100%|██████████| 1/1 [00:00<00:00, 110.93it/s]

Retrieved 3 documents (after filtering)


Answer: [{'type': 'text', 'text': 'Based on the context, YOLOv5 is a **real-time detection model** that was tested with 22 classes over 100 epochs. It achieved a confidence curve of 69.4% and a precision level of 92.8%, producing smoother validation curves than the YOLOv8 model.', 'extras': {'signature': 'EvUMCvIMAb4+9vuxxN3SPA7/LEbO4keCX/sZz83jNG7aPv1MeXF5fYkIu7jbROy+AFPu+d5OjwjN67/eW6Dx5E+f7RQnAuJTYSaWLz/KZJWvQ8V+x4ejA5Nf2REnkzOLkawbJBhGiIS/3/2gOSY/7plV5F0SJiXaQctfVRxeZApEAGmc9LCl7vYitzQzD5TvG325PwtDJO7n072WhYWiHL6v1scyI/Hto+pwN0lqggPu2dLiCfDW17HvIrhhT8pSNOa1EC4PRzGm5NWyIP/fQQdPKfV1UbGJxYxZn8XcQh6p6gAGrF+BxJ7Jax5s2uBPo/ZJppDwI1ITw3UsaKvl7C0WwZAEYqm8fHOBce2fqCiNx3AfAV/PYSqsH5c3GhsH99MmJU9dCiu8/5VxaEUsImKlcNLA5OjCIhqNR8l8qGgslNpHjQH+sEkWiCQNksRl7OOwYo2T3Abrc4SK9679U5D0qV3L4MvdF8EBudwbWnvK1V+5vAnoCBNwoggtjPv0zZVqRCW42vn/R+KMvk2a74cMzeJdbuHDUBxzXhQjhboQ6IkExycsqmW8Tu0nvyL6yBlPkAYuVwU6wchSC1rGOPTR7oJRS3VxruMbYEqvjsOBsk7pv0JWQdkIOFvyLdpoxIhT/sAGsFQ8iJIebC9Brp+197Mw/UGr3G3p+6fmbDlCYkd1gdWSA